# Chunk 12 — Fine-Tune Data Preparation (v2.0)

Build PyG graphs for the fine-tuning grids (35×35, 45×45, 55×55), construct the manifest and splits, then define the `FinetuneDataset` with mandatory z-score normalization at load time.

**Normalization contract (non-negotiable):** On-disk `data.y` is RAW dB. `FinetuneDataset` z-scores at load using the 25×25 training-split `s11_mean` / `s11_std` and preserves `y_raw`. `evaluate()` de-normalizes with the same statistics. Never recompute statistics from the fine-tune pool.

## 1. Environment Setup & Dependencies

Install the required Python packages for running the notebook.

In [1]:
# Cell 1 — Install dependencies
!pip install scipy numpy matplotlib torch torchvision \
    torch-geometric tqdm scikit-learn pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.2 MB/s eta 0:00:00


## 2. Repository Setup

Clone the GitHub repository to the local Colab environment and add `src` to the Python path.

In [2]:
# Cell 2 — Clone repo (re-clones every session; pulls latest if already exists)
import os
REPO_ROOT = '/content/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')   # makes 'from model import AntennaGNN' work
print(f'Repo ready at {REPO_ROOT}')

Cloning into '/content/antenna-gnn'...
remote: Enumerating objects: 261, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 261 (delta 137), reused 223 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (261/261), 5.42 MiB | 18.43 MiB/s, done.
Resolving deltas: 100% (137/137), done.
Repo ready at /content/antenna-gnn


## 3. Drive Mount and Path Configuration

Mount Google Drive to access `RAW_DATA` and store the computed `.pt` cache files in `DATA_ROOT`. This ensures our generated artifacts are preserved beyond the lifespan of the Colab session.

In [3]:
# Cell 3 — Mount Drive and set data paths
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'
for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints',
          f'{DATA_ROOT}/figures',   f'{DATA_ROOT}/splits',
          f'{DATA_ROOT}/data/processed', f'{DATA_ROOT}/data/processed_finetune']:
    os.makedirs(d, exist_ok=True)
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')

Mounted at /content/drive
Drive mounted. DATA_ROOT=/content/drive/MyDrive/antenna_gnn


---
## CELL A — `build_pyg_graph` (frozen)

Verbatim copy of the graph construction function from Chunk 5. Node features are `[metal, x_norm, y_norm, is_seed, dist_to_seed_centroid]`; the virtual global node is `[0, 0.5, 0.5, -1.0, 0.0]`; 4-connectivity edges carry `[edge_type, direction]` with `etype_map {(1,1):0, (1,0):1, (0,1):2, (0,0):3}` and virtual edges typed `[4, 4]`.

The target `y` is stored as **RAW dB** on disk — the `FinetuneDataset` z-scores at load time per the normalization contract.

In [4]:
# CELL A — build_pyg_graph (frozen, verbatim from Chunk 5)
import torch
from torch_geometric.data import Data
import numpy as np

def build_pyg_graph(patch_pattern, s11_db, seed_mask, N):
    # Compute seed centroid
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)

    # Node features: (N*N + 1) nodes, 5 features each
    node_feats = []
    for i in range(N):
        for j in range(N):
            metal    = float(patch_pattern[i, j])
            x_norm   = j / (N - 1)
            y_norm   = i / (N - 1)
            is_seed  = float(seed_mask[i, j])
            dist_f   = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])

    # Virtual global node (index N*N): all zeros except placeholder (is_seed=-1 for virtual node)
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float)

    # 4-connectivity edges
    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0),(0,-1,1),(-1,0,2),(1,0,3)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])

    # Virtual node edges (connect to all metal pixels only)
    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]  # virtual edge type

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)

    # Target
    # RAW dB on disk — Dataset z-scores at load (see normalization contract)
    y = torch.tensor(s11_db, dtype=torch.float).unsqueeze(0)  # (1, 201)

    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, y=y)

---
## CELL B — Graph Construction for 35×35, 45×45, 55×55

For each grid size: glob the raw `.mat` files from `RAW_DATA`, load the seed mask, bulk-copy files to local Colab disk using a `ThreadPoolExecutor` (Drive random-access is the bottleneck), then build and save `DATA_ROOT/data/processed_finetune/{N}x{N}/sample_{i}.pt`.

Each `Data` object is tagged with `grid_size`, `pixel_size_mm = 32.375 / N`, and `is_functioning`. Files that already exist are skipped. A `{N}x{N}_DONE.txt` sentinel is written upon completion.

In [5]:
# CELL B — Graph construction for 35x35, 45x45, 55x55
import glob, os, torch, shutil, concurrent.futures
import scipy.io as sio
import numpy as np
from tqdm.auto import tqdm

for N in [35, 45, 55]:
    done_marker = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}_DONE.txt'
    if os.path.exists(done_marker):
        print(f'Grid {N}x{N} already processed (sentinel found). Skipping.')
        continue

    raw_files = sorted(glob.glob(
        f'{RAW_DATA}/fine-tuning dataset/{N}x{N}/**/Mat_Files/*.mat',
        recursive=True))
    seed_mask = np.load(f'{DATA_ROOT}/artifacts/seed_mask_{N}.npy')

    proc_dir = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}'
    os.makedirs(proc_dir, exist_ok=True)

    print(f'\nProcessing {N}x{N} grid, {len(raw_files)} files...')

    # ── Bulk-copy to local disk (Drive random-access is the bottleneck) ──
    local_dir = f'/content/raw_{N}x{N}'
    os.makedirs(local_dir, exist_ok=True)

    print('Bulk copying files to local disk...')
    def copy_file(src):
        dst = os.path.join(local_dir, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        return dst

    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        local_paths = list(tqdm(
            executor.map(copy_file, raw_files),
            total=len(raw_files),
            desc=f'Copying {N}x{N} files'))

    # ── Build PyG graphs ──
    functioning_count = 0
    for i, local_f in enumerate(tqdm(local_paths, desc=f'Processing {N}x{N} graphs')):
        proc_path = f'{proc_dir}/sample_{i}.pt'

        if os.path.exists(proc_path):
            data = torch.load(proc_path, weights_only=False)
            functioning_count += getattr(data, 'is_functioning', 0)
        else:
            mat = sio.loadmat(local_f)
            is_functioning = int(mat['resonant_freqs'].size > 0)
            functioning_count += is_functioning

            data = build_pyg_graph(
                mat['patch_pattern'], mat['S11_dB'].flatten(),
                seed_mask, N)
            data.grid_size = N
            data.pixel_size_mm = 32.375 / N
            data.is_functioning = is_functioning
            torch.save(data, proc_path)

    # ── Cleanup local cache ──
    print(f'Cleaning up local cache {local_dir}...')
    shutil.rmtree(local_dir)

    # ── Sentinel ──
    with open(done_marker, 'w') as fh:
        fh.write('DONE\n')

    func_pct = functioning_count / len(raw_files) * 100
    print(f'Grid {N}x{N}: {len(raw_files)} total, {func_pct:.1f}% functioning')

print('\n(Expected approximately 63% / 65% / 48%)')

Grid 35x35 already processed (sentinel found). Skipping.
Grid 45x45 already processed (sentinel found). Skipping.
Grid 55x55 already processed (sentinel found). Skipping.

(Expected approximately 63% / 65% / 48%)


---
## CELL C — Manifest

Build `DATA_ROOT/artifacts/finetune_manifest.csv` with columns `grid_size, sample_idx, is_functioning, pixel_size_mm`. If it already exists, load it instead. This manifest is the **AUTHORITATIVE** source of `is_functioning` for every downstream chunk.

In [6]:
# CELL C — Manifest
import glob
import pandas as pd
from tqdm.auto import tqdm

manifest_path = f'{DATA_ROOT}/artifacts/finetune_manifest.csv'

if os.path.exists(manifest_path):
    print('Loading existing manifest...')
    manifest = pd.read_csv(manifest_path)
else:
    print('Building manifest from .pt files...')
    records = []
    for N in [35, 45, 55]:
        proc_dir = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}'
        pt_files = sorted(glob.glob(f'{proc_dir}/sample_*.pt'))

        for pt_file in tqdm(pt_files, desc=f'Scanning {N}x{N}'):
            idx = int(os.path.basename(pt_file).split('_')[1].split('.')[0])
            data = torch.load(pt_file, weights_only=False)
            records.append({
                'grid_size': N,
                'sample_idx': idx,
                'is_functioning': int(data.is_functioning),
                'pixel_size_mm': float(data.pixel_size_mm),
            })
    manifest = pd.DataFrame(records)
    manifest.to_csv(manifest_path, index=False)
    print(f'Saved manifest to {manifest_path}')

print(f'\nManifest: {len(manifest)} samples')
for N in [35, 45, 55]:
    grid_data = manifest[manifest['grid_size'] == N]
    func = grid_data['is_functioning'].sum()
    print(f'  {N}x{N}: {len(grid_data)} total, {func} functioning ({func/len(grid_data)*100:.1f}%)')

Loading existing manifest...

Manifest: 14964 samples
  35x35: 4988 total, 3163 functioning (63.4%)
  45x45: 6984 total, 4559 functioning (65.3%)
  55x55: 2992 total, 1448 functioning (48.4%)


---
## CELL D — Splits

Read `indices.json` from Chunk 5. If it already contains val/test entries for grids 35/45/55, reuse them directly (**Case A**). Otherwise treat Chunk 5's train entries as the pool, everything else as test, and carve a 15% validation set out of the pool with `train_test_split` stratified on `f'{grid}_{is_functioning}'` (**Case B**).

Writes `finetune_pool_indices.json`, `finetune_val_indices.json`, `finetune_test_indices.json`. Backs up any pre-existing files to `*_legacy_independent_split.json`.

In [7]:
# CELL D — Splits
import json
import shutil
from sklearn.model_selection import train_test_split

# ── Load Chunk 5 splits ──
with open(f'{DATA_ROOT}/splits/indices.json', 'r') as f:
    c5_splits = json.load(f)

print('Chunk 5 indices.json keys:', list(c5_splits.keys()))

# ── Check if Chunk 5 has val/test entries for fine-tune grids ──
has_val_test = (
    any(entry[0] in [35, 45, 55] for entry in c5_splits.get('val', [])) or
    any(entry[0] in [35, 45, 55] for entry in c5_splits.get('test', []))
)

# ── Build manifest index for stratification ──
manifest_indexed = manifest.set_index(['grid_size', 'sample_idx'])
pool_list = list(zip(manifest['grid_size'].astype(int),
                     manifest['sample_idx'].astype(int)))

# ── Backup pre-existing splits ──
for split_name in ['finetune_test_indices.json',
                   'finetune_val_indices.json',
                   'finetune_pool_indices.json']:
    src_path = f'{DATA_ROOT}/splits/{split_name}'
    dst_path = f'{DATA_ROOT}/splits/{split_name.replace(".json", "_legacy_independent_split.json")}'
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        shutil.copy(src_path, dst_path)
        print(f'Backed up {split_name} to legacy independent split.')

if has_val_test:
    # ── Case A: Reuse Chunk 5's existing train/val/test ──
    print('\nCase A: Using existing train/val/test splits for fine-tune grids from Chunk 5.')
    train_pool = [list(x) for x in c5_splits.get('train', []) if x[0] in [35, 45, 55]]
    val_set    = [list(x) for x in c5_splits.get('val', [])   if x[0] in [35, 45, 55]]
    test_set   = [list(x) for x in c5_splits.get('test', [])  if x[0] in [35, 45, 55]]
else:
    # ── Case B: Derive from Chunk 5 train; everything else is test ──
    print('\nCase B: Chunk 5 only has train partition for fine-tune grids. Deriving val/test...')
    c5_train_ft = set(tuple(x) for x in c5_splits.get('train', [])
                      if x[0] in [35, 45, 55])

    test_set = [list(x) for x in pool_list if tuple(x) not in c5_train_ft]
    c5_train_list = [list(x) for x in pool_list if tuple(x) in c5_train_ft]

    c5_train_labels = [
        f"{x[0]}_{manifest_indexed.loc[(x[0], x[1]), 'is_functioning']}"
        for x in c5_train_list
    ]

    train_pool, val_set = train_test_split(
        c5_train_list, test_size=0.15,
        stratify=c5_train_labels, random_state=42
    )

# ── Write splits ──
with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json', 'w') as f:
    json.dump(train_pool, f)
with open(f'{DATA_ROOT}/splits/finetune_val_indices.json', 'w') as f:
    json.dump(val_set, f)
with open(f'{DATA_ROOT}/splits/finetune_test_indices.json', 'w') as f:
    json.dump(test_set, f)

# ── Assert pairwise disjoint ──
set_pool = set(tuple(x) for x in train_pool)
set_val  = set(tuple(x) for x in val_set)
set_test = set(tuple(x) for x in test_set)

assert set_pool.isdisjoint(set_val),  'Pool and Val splits overlap!'
assert set_val.isdisjoint(set_test),  'Val and Test splits overlap!'
assert set_pool.isdisjoint(set_test), 'Pool and Test splits overlap!'

# ── Assert union = manifest ──
assert len(train_pool) + len(val_set) + len(test_set) == len(manifest), (
    f'Union of splits ({len(train_pool)+len(val_set)+len(test_set)}) '
    f'!= manifest ({len(manifest)})'
)

print(f'\n✓ Splits are pairwise disjoint; union = {len(manifest)} = manifest size')

# ── Per-grid breakdown with functioning counts ──
for name, dataset in [('Pool', train_pool), ('Val', val_set), ('Test', test_set)]:
    print(f'\n{name} set breakdown:')
    for N in [35, 45, 55]:
        grid_items = [x for x in dataset if x[0] == N]
        count = len(grid_items)
        if count > 0:
            func = sum(int(manifest_indexed.loc[(N, idx), 'is_functioning'])
                       for _, idx in grid_items)
        else:
            func = 0
        print(f'  {N}x{N}: {count} (functioning: {func})')

Chunk 5 indices.json keys: ['train', 'val', 'test']

Case A: Using existing train/val/test splits for fine-tune grids from Chunk 5.

✓ Splits are pairwise disjoint; union = 14964 = manifest size

Pool set breakdown:
  35x35: 3990 (functioning: 2530)
  45x45: 5587 (functioning: 3647)
  55x55: 2394 (functioning: 1159)

Val set breakdown:
  35x35: 499 (functioning: 316)
  45x45: 698 (functioning: 456)
  55x55: 299 (functioning: 145)

Test set breakdown:
  35x35: 499 (functioning: 317)
  45x45: 699 (functioning: 456)
  55x55: 299 (functioning: 144)


---
## CELL E — Load the 25×25 Statistics

Load the 25×25 **training-split** normalization statistics. These are the canonical statistics — do **NOT** recompute them from the fine-tune pool under any circumstances. The pretrained model's output head is calibrated to these values.

In [8]:
# CELL E — Load the 25x25 statistics
import numpy as np
import torch

s11_mean_np = np.load(f'{DATA_ROOT}/artifacts/s11_mean.npy')
s11_std_np  = np.load(f'{DATA_ROOT}/artifacts/s11_std.npy')

s11_mean_cpu = torch.tensor(s11_mean_np, dtype=torch.float32)
s11_std_cpu  = torch.tensor(s11_std_np,  dtype=torch.float32)

assert s11_mean_cpu.shape == (201,), f'Expected shape (201,), got {s11_mean_cpu.shape}'
assert s11_std_cpu.shape  == (201,), f'Expected shape (201,), got {s11_std_cpu.shape}'

print('25x25 training-split statistics (canonical, DO NOT recompute):')
print(f'  s11_mean: mean={s11_mean_np.mean():.4f}, std={s11_mean_np.std():.4f}')
print(f'  s11_std:  mean={s11_std_np.mean():.4f}, std={s11_std_np.std():.4f}')

25x25 training-split statistics (canonical, DO NOT recompute):
  s11_mean: mean=-1.2196, std=1.5749
  s11_std:  mean=1.3767, std=1.9057


---
## CELL F — `FinetuneDataset` with Load-Time Normalization

The `FinetuneDataset` class implements the **normalization contract**:
- On-disk `data.y` is raw dB.
- At load time, `y_raw` is preserved and `y` is z-scored using the 25×25 training-split statistics.
- Statistics must not be `None` — the constructor asserts this.
- `torch.load` uses `weights_only=False`.

In [9]:
# CELL F — FinetuneDataset with load-time normalization
from torch.utils.data import Dataset as TorchDataset

class FinetuneDataset(TorchDataset):
    """Fine-tune graph dataset with mandatory z-score normalization at load.

    On-disk data.y is raw dB. This dataset z-scores y at load time using
    the 25x25 training-split s11_mean / s11_std, and preserves y_raw.
    evaluate() de-normalizes with the same statistics.

    Never pass s11_mean=None or s11_std=None.
    """

    def __init__(self, indices, processed_dir_base, s11_mean, s11_std):
        assert s11_mean is not None, 'FinetuneDataset requires s11_mean (got None)'
        assert s11_std is not None,  'FinetuneDataset requires s11_std (got None)'
        self.indices = indices
        self.processed_dir_base = processed_dir_base
        self.s11_mean = s11_mean   # (201,) tensor
        self.s11_std  = s11_std    # (201,) tensor

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        grid_size, local_idx = self.indices[idx]
        path = f'{self.processed_dir_base}/{grid_size}x{grid_size}/sample_{local_idx}.pt'
        data = torch.load(path, weights_only=False)

        # Preserve raw dB target, then z-score
        data.y_raw = data.y.clone()                              # (1, 201) raw dB
        data.y = (data.y - self.s11_mean) / (self.s11_std + 1e-8)  # (1, 201) normalized

        return data

# Construct the base directory path for all downstream loaders
processed_dir = f'{DATA_ROOT}/data/processed_finetune'
print(f'FinetuneDataset defined. processed_dir_base = {processed_dir}')

FinetuneDataset defined. processed_dir_base = /content/drive/MyDrive/antenna_gnn/data/processed_finetune


---
## CELL G — Normalization Guard (MANDATORY)

Run the canonical guard on a 512-sample probe of the pool:
- Print mean / std / min / max of the normalized targets.
- Assert `abs(mean) < 0.15`, `abs(std - 1.0) < 0.25`, `min > -12`.

Then verify the round trip on a single sample: de-normalizing `y` must recover `y_raw` to within `1e-4`.

In [10]:
# CELL G — Normalization guard (CORRECTED)
# Assert IDENTITIES, not physical ranges. Per-point s11_std spans 0.0139-7.1613,
# so legitimately normalized values can reach ~-84 where std is smallest. Any
# range-based bound on normalized y therefore overlaps the raw-dB range and
# cannot discriminate. The round trip can.
from torch_geometric.loader import DataLoader
import json
import numpy as np

with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json') as f:
    pool_indices = json.load(f)

# Random probe, not the first 512 — the split file is grid-ordered.
rng = np.random.default_rng(42)
probe_ids = [pool_indices[i] for i in
             rng.choice(len(pool_indices), size=min(512, len(pool_indices)),
                        replace=False)]
probe_ds = FinetuneDataset(probe_ids, processed_dir, s11_mean_cpu, s11_std_cpu)
probe_loader = DataLoader(probe_ds, batch_size=64, shuffle=False)

ys, yr = [], []
for b in probe_loader:
    ys.append(b.y.view(-1, 201))
    yr.append(b.y_raw.view(-1, 201))
all_y, all_raw = torch.cat(ys), torch.cat(yr)

# ---- ASSERT: identities ----
assert hasattr(probe_ds[0], 'y_raw'), "y_raw missing — Dataset did not normalize"
assert not torch.allclose(all_y, all_raw), "y == y_raw — normalization is a no-op"
recon = all_y * s11_std_cpu + s11_mean_cpu
maxdiff = (recon - all_raw).abs().max().item()
assert maxdiff < 1e-3, f"round trip FAILED: max diff {maxdiff:.6f}"
print(f"✓ round trip exact over {len(all_y)} samples (max diff {maxdiff:.2e})")

# ---- ASSERT: raw side is physical dB ----
assert all_raw.max() <= 0.01, f"raw max {all_raw.max():.3f} > 0 dB"
assert -200 < all_raw.min() < 0, f"raw min {all_raw.min():.2f} implausible"
print(f"✓ raw dB physical: min={all_raw.min():.2f} max={all_raw.max():.4f} dB")

# ---- REPORT: normalized distribution, loose sanity only ----
print(f"\nnormalized y: mean={all_y.mean():.4f} std={all_y.std():.4f} "
      f"min={all_y.min():.2f} max={all_y.max():.2f}")
print("  (mean need NOT be 0: stats are from the 25x25 split, not this pool)")
assert abs(all_y.mean()) < 0.6 and abs(all_y.std() - 1.0) < 0.6, \
    "normalized distribution far from unit scale — investigate"

# ---- DIAGNOSTIC: where do the extremes come from? ----
freq_axis = np.linspace(1.0, 4.0, 201)
per_pt_std = all_y.std(dim=0).numpy()
worst = int(per_pt_std.argmax())
print(f"\nper-point normalized std: min={per_pt_std.min():.2f} "
      f"median={np.median(per_pt_std):.2f} max={per_pt_std.max():.2f}")
print(f"  widest at index {worst} ({freq_axis[worst]:.2f} GHz); "
      f"s11_std there = {s11_std_np[worst]:.4f}")

bad_pts = np.where(per_pt_std > 5)[0]
print(f"\n{len(bad_pts)} of 201 frequency points have normalized std > 5:")
if len(bad_pts):
    print(f"  indices {bad_pts.tolist()[:12]}"
          f"{' ...' if len(bad_pts) > 12 else ''}")
    print(f"  freqs   {[round(float(freq_axis[i]), 2) for i in bad_pts[:12]]}")
    print(f"  s11_std {[round(float(s11_std_np[i]), 4) for i in bad_pts[:12]]}")
frac = (all_y.abs() > 15).float().mean().item()
print(f"\nfraction of all (sample, freq) targets with |y| > 15: {frac:.4%}")
print(f"MSE contribution of the single worst point: "
      f"{(all_y.max().item() ** 2):.0f} vs ~1 at a typical point")

✓ round trip exact over 512 samples (max diff 1.91e-06)
✓ raw dB physical: min=-36.43 max=-0.0000 dB

normalized y: mean=0.1737 std=1.1037 min=-46.22 max=3.53
  (mean need NOT be 0: stats are from the 25x25 split, not this pool)

per-point normalized std: min=0.53 median=1.04 max=2.30
  widest at index 26 (1.39 GHz); s11_std there = 0.0326

0 of 201 frequency points have normalized std > 5:

fraction of all (sample, freq) targets with |y| > 15: 0.0360%
MSE contribution of the single worst point: 12 vs ~1 at a typical point


---
## CELL H — Sanity Checks

Build DataLoaders for pool / val / test. Print one batch showing grid sizes, NaN checks, and `y` shape. Spot-check five test samples printing `grid_size` and `is_functioning`. Assert no NaNs anywhere.

In [11]:
# CELL H — Sanity checks
import json
from torch_geometric.loader import DataLoader

with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json', 'r') as f:
    pool_indices = json.load(f)
with open(f'{DATA_ROOT}/splits/finetune_val_indices.json', 'r') as f:
    val_indices = json.load(f)
with open(f'{DATA_ROOT}/splits/finetune_test_indices.json', 'r') as f:
    test_indices = json.load(f)

pool_ds = FinetuneDataset(pool_indices, processed_dir, s11_mean_cpu, s11_std_cpu)
val_ds  = FinetuneDataset(val_indices,  processed_dir, s11_mean_cpu, s11_std_cpu)
test_ds = FinetuneDataset(test_indices, processed_dir, s11_mean_cpu, s11_std_cpu)

print(f'Dataset sizes:  pool={len(pool_ds)}  val={len(val_ds)}  test={len(test_ds)}')

# ── Print one batch ──
pool_loader = DataLoader(pool_ds, batch_size=16, shuffle=True)
batch = next(iter(pool_loader))

print(f'\nBatch from Pool:')
print(f'  num_graphs = {batch.num_graphs}')
print(f'  x.shape    = {batch.x.shape}')
print(f'  y.shape    = {batch.y.shape}')
print(f'  grid_sizes = {batch.grid_size.tolist()}')

assert not torch.isnan(batch.x).any(), 'NaN detected in batch.x!'
assert not torch.isnan(batch.y).any(), 'NaN detected in batch.y!'
print(f'  NaNs in x: {torch.isnan(batch.x).any().item()}')
print(f'  NaNs in y: {torch.isnan(batch.y).any().item()}')

# ── Spot-check five test samples ──
print(f'\nSpot check (5 test samples):')
for i in range(min(5, len(test_ds))):
    sample = test_ds[i]
    assert not torch.isnan(sample.x).any(), f'NaN in test sample {i} x!'
    assert not torch.isnan(sample.y).any(), f'NaN in test sample {i} y!'
    print(f'  Sample {i}: grid_size={sample.grid_size}, '
          f'is_functioning={sample.is_functioning}, '
          f'y.shape={sample.y.shape}')

print(f'\n✓ All sanity checks passed.')

Dataset sizes:  pool=11971  val=1496  test=1497

Batch from Pool:
  num_graphs = 16
  x.shape    = torch.Size([30416, 5])
  y.shape    = torch.Size([16, 201])
  grid_sizes = [35, 45, 35, 45, 55, 45, 45, 55, 45, 35, 45, 45, 35, 45, 45, 35]
  NaNs in x: False
  NaNs in y: False

Spot check (5 test samples):
  Sample 0: grid_size=45, is_functioning=1, y.shape=torch.Size([1, 201])
  Sample 1: grid_size=45, is_functioning=1, y.shape=torch.Size([1, 201])
  Sample 2: grid_size=45, is_functioning=0, y.shape=torch.Size([1, 201])
  Sample 3: grid_size=45, is_functioning=1, y.shape=torch.Size([1, 201])
  Sample 4: grid_size=45, is_functioning=1, y.shape=torch.Size([1, 201])

✓ All sanity checks passed.


In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL I — SMOKE TEST part 1: zero-shot scale + label reconciliation
# Validation only. Does NOT touch the test set. Inference only, no training.
# ═══════════════════════════════════════════════════════════════════════════
import json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from scipy.signal import find_peaks
from tqdm.auto import tqdm
from model import AntennaGNN

if 'device' not in dir():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device = {device}")

s11_mean_dev = s11_mean_cpu.to(device)
s11_std_dev  = s11_std_cpu.to(device)
freq_axis = np.linspace(1.0, 4.0, 201)

with open(f'{DATA_ROOT}/splits/finetune_val_indices.json') as f:
    val_indices = json.load(f)
with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json') as f:
    pool_indices = json.load(f)

val_ds = FinetuneDataset(val_indices, processed_dir, s11_mean_cpu, s11_std_cpu)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)   # shuffle=False: pos alignment


def load_pretrained_gnn(path, device):
    m = AntennaGNN()
    ck = torch.load(path, map_location=device, weights_only=False)
    m.load_state_dict(ck['model_state'])
    return m.to(device)


def extract_resonant_freq(s11_db, freq_axis_ghz, threshold_db=-10):
    peaks, _ = find_peaks(-s11_db, height=-threshold_db, distance=5)
    if len(peaks) == 0:
        return None
    return freq_axis_ghz[peaks[np.argmax(-s11_db[peaks])]]


def smoke_evaluate(model, loader):
    """Per-grid metrics + per-sample records. Mirrors Chunk 13's evaluate."""
    model.eval()
    agg = {g: {'s11': [], 'n_total': 0, 'n_true': 0, 'n_pred': 0,
               'n_both': 0, 'freq': []} for g in (35, 45, 55)}
    per_sample, pos = [], 0
    with torch.no_grad():
        for batch in tqdm(loader, desc='eval', leave=False):
            batch = batch.to(device)
            out_db  = model(batch).view(-1, 201) * s11_std_dev + s11_mean_dev
            true_db = batch.y.view(-1, 201)     * s11_std_dev + s11_mean_dev
            mae = (out_db - true_db).abs().mean(dim=1)
            for i in range(out_db.shape[0]):
                g = int(batch.grid_size[i].item())
                p = out_db[i].cpu().numpy()
                t = true_db[i].cpu().numpy()
                pr, tr = (extract_resonant_freq(p, freq_axis),
                          extract_resonant_freq(t, freq_axis))
                pf, tf = pr is not None, tr is not None
                per_sample.append({'pos': pos, 'grid': g,
                                   'true_func': tf, 'pred_func': pf,
                                   'true_s11_min': float(t.min()),
                                   'pred_s11_min': float(p.min()),
                                   'pred_s11_max': float(p.max())})
                pos += 1
                if g in agg:
                    a = agg[g]
                    a['s11'].append(mae[i].item()); a['n_total'] += 1
                    a['n_true'] += tf; a['n_pred'] += pf
                    if pf and tf:
                        a['n_both'] += 1; a['freq'].append(abs(pr - tr))
    assert pos == len(loader.dataset), f"pos {pos} != {len(loader.dataset)}"
    res = {g: {'s11_mae': float(np.mean(a['s11'])) if a['s11'] else 0.0,
               'freq_mae': float(np.mean(a['freq'])) if a['freq'] else float('nan'),
               'n_total': a['n_total'], 'n_true_func': a['n_true'],
               'n_pred_func': a['n_pred'], 'n_both': a['n_both']}
           for g, a in agg.items()}
    return res, per_sample


print("\n" + "=" * 74)
print("CHECK 1 — zero-shot on validation (v1.0 gave 3.91 / 4.03 / 3.65)")
print("=" * 74)
base = load_pretrained_gnn(f'{DATA_ROOT}/checkpoints/best_model.pt', device)
zs, ps = smoke_evaluate(base, val_loader)
for g in (35, 45, 55):
    r = zs[g]
    print(f"  {g}x{g}: S11 MAE = {r['s11_mae']:.4f} dB   "
          f"freq MAE = {r['freq_mae']:.4f} GHz   "
          f"detect = {r['n_pred_func']}/{r['n_total']} "
          f"({r['n_pred_func']/r['n_total']:.1%})")
zs_weighted = sum(zs[g]['s11_mae'] * zs[g]['n_total'] for g in (35, 45, 55)) \
              / sum(zs[g]['n_total'] for g in (35, 45, 55))
print(f"  weighted: {zs_weighted:.4f} dB")
print("  PASS if clearly below ~3.9. Still near 4 => contract broken downstream.")

print("\n" + "=" * 74)
print("CHECK 2 — ground-truth resonance depths must be physical")
print("=" * 74)
tmin = np.array([r['true_s11_min'] for r in ps])
pmin = np.array([r['pred_s11_min'] for r in ps])
pmax = np.array([r['pred_s11_max'] for r in ps])
print(f"  min(true S11): median={np.median(tmin):7.2f}  p10={np.percentile(tmin,10):7.2f}  "
      f"min={tmin.min():7.2f}  max={tmin.max():7.2f} dB")
print(f"  min(pred S11): median={np.median(pmin):7.2f}  p10={np.percentile(pmin,10):7.2f} dB")
print(f"  max(pred S11): median={np.median(pmax):7.2f}  max={pmax.max():7.2f} dB "
      f"(a passive antenna cannot exceed 0)")
print("  PASS if true median is roughly -8 to -20. Near -87 => contract broken.")

print("\n" + "=" * 74)
print("CHECK 3 — find_peaks labels vs manifest is_functioning (validation)")
print("=" * 74)
mf = pd.read_csv(f'{DATA_ROOT}/artifacts/finetune_manifest.csv')
pos_map = pd.DataFrame([(i, g, s) for i, (g, s) in enumerate(val_indices)],
                       columns=['pos', 'grid_size', 'sample_idx'])
rec = pos_map.merge(pd.DataFrame(ps)[['pos', 'grid', 'true_func', 'true_s11_min']],
                    on='pos')
assert (rec['grid_size'] == rec['grid']).all(), "pos -> val_indices misaligned"
rec = rec.merge(mf[['grid_size', 'sample_idx', 'is_functioning']],
                on=['grid_size', 'sample_idx'], how='left')
assert rec['is_functioning'].notna().all(), "val sample missing from manifest"
rec['manifest_func'] = rec['is_functioning'].astype(int) == 1

print(f"  {'Grid':<8}{'n':>6}{'find_peaks':>12}{'manifest':>11}{'agree':>9}")
for g in (35, 45, 55):
    s = rec[rec['grid_size'] == g]
    print(f"  {str(g)+'x'+str(g):<8}{len(s):>6}{s['true_func'].mean():>11.1%}"
          f"{s['manifest_func'].mean():>11.1%}"
          f"{(s['true_func'] == s['manifest_func']).mean():>9.1%}")
agree = (rec['true_func'] == rec['manifest_func']).mean()
print(f"  {'ALL':<8}{len(rec):>6}{rec['true_func'].mean():>11.1%}"
      f"{rec['manifest_func'].mean():>11.1%}{agree:>9.1%}")
print("\n  confusion (rows=manifest, cols=find_peaks):")
print(pd.crosstab(rec['manifest_func'], rec['true_func'],
                  rownames=['manifest'], colnames=['find_peaks']).to_string())

# Expected residual: band-edge dips, where find_peaks needs a rising side.
dis = rec[rec['true_func'] != rec['manifest_func']]
if len(dis):
    print(f"\n  {len(dis)} disagreements; min(true S11) median "
          f"{dis['true_s11_min'].median():.2f} dB")
    edge = ((dis['true_s11_min'] > -11) | (dis['true_s11_min'] < -11)).sum()  # placeholder
    near = (dis['true_s11_min'].between(-12, -8)).mean()
    print(f"  {near:.0%} sit within -12..-8 dB, i.e. borderline at the threshold")
print(f"\n  PASS if agreement >= 97%.")

del base
torch.cuda.empty_cache()

device = cuda

CHECK 1 — zero-shot on validation (v1.0 gave 3.91 / 4.03 / 3.65)


eval:   0%|          | 0/24 [00:00<?, ?it/s]

  35x35: S11 MAE = 0.5708 dB   freq MAE = 0.0265 GHz   detect = 190/499 (38.1%)
  45x45: S11 MAE = 0.6967 dB   freq MAE = 0.0481 GHz   detect = 300/698 (43.0%)
  55x55: S11 MAE = 0.7818 dB   freq MAE = 0.0840 GHz   detect = 129/299 (43.1%)
  weighted: 0.6717 dB
  PASS if clearly below ~3.9. Still near 4 => contract broken downstream.

CHECK 2 — ground-truth resonance depths must be physical
  min(true S11): median= -11.85  p10= -20.49  min= -43.74  max=  -0.01 dB
  min(pred S11): median=  -4.51  p10= -25.22 dB
  max(pred S11): median=  -0.05  max=   0.32 dB (a passive antenna cannot exceed 0)
  PASS if true median is roughly -8 to -20. Near -87 => contract broken.

CHECK 3 — find_peaks labels vs manifest is_functioning (validation)
  Grid         n  find_peaks   manifest    agree
  35x35      499      63.3%      63.3%   100.0%
  45x45      698      65.3%      65.3%   100.0%
  55x55      299      48.5%      48.5%   100.0%
  ALL       1496      61.3%      61.3%   100.0%

  confusion (row

In [13]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL J — SMOKE TEST part 2: loss scale and gradient-norm headroom
# 5 epochs, Strategy A (full fine-tune, lr=1e-4), 2,000-sample pool subset
# to match Chunk 13's screen so the loss numbers are directly comparable.
# ═══════════════════════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split
from torch.optim import Adam

print("=" * 74)
print("CHECK 4 — loss scale and gradient clipping headroom")
print("=" * 74)

sub, _ = train_test_split(pool_indices, train_size=2000,
                          stratify=[str(x[0]) for x in pool_indices],
                          random_state=42)
train_loader = DataLoader(
    FinetuneDataset(sub, processed_dir, s11_mean_cpu, s11_std_cpu),
    batch_size=32, shuffle=True)

model = load_pretrained_gnn(f'{DATA_ROOT}/checkpoints/best_model.pt', device)
opt = Adam(filter(lambda p: p.requires_grad, model.parameters()),
           lr=1e-4, weight_decay=1e-4)
criterion = nn.MSELoss()

all_norms, first_loss = [], None
for epoch in range(5):
    model.train()
    ep_losses, ep_norms = [], []
    for step, batch in enumerate(tqdm(train_loader, desc=f'epoch {epoch+1}/5',
                                      leave=False)):
        batch = batch.to(device)
        opt.zero_grad()
        loss = criterion(model(batch).view(-1, 201), batch.y.view(-1, 201))
        loss.backward()
        # clip_grad_norm_ RETURNS the pre-clip total norm
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        ep_losses.append(loss.item()); ep_norms.append(float(gn))
        if epoch == 0 and step == 0:
            first_loss = loss.item()
    all_norms += ep_norms
    clipped = np.mean(np.array(ep_norms) > 1.0)
    print(f"  epoch {epoch+1}: train MSE mean={np.mean(ep_losses):.4f} "
          f"first={ep_losses[0]:.4f} last={ep_losses[-1]:.4f} | "
          f"grad-norm median={np.median(ep_norms):.3f} "
          f"clipped={clipped:.0%}")

norms = np.array(all_norms)
print(f"\n  first-ever step MSE : {first_loss:.4f}   "
      f"(PASS if order ~1, not hundreds)")
print(f"  grad-norm median    : {np.median(norms):.3f}   "
      f"(PASS if <= ~1.0)")
print(f"  grad-norm p90 / max : {np.percentile(norms,90):.3f} / {norms.max():.3f}")
print(f"  steps hitting clip  : {(norms > 1.0).mean():.1%}   "
      f"(PASS if well under ~50%; near 100% => throttled, as in v1.0)")

post, _ = smoke_evaluate(model, val_loader)
post_w = sum(post[g]['s11_mae'] * post[g]['n_total'] for g in (35, 45, 55)) \
         / sum(post[g]['n_total'] for g in (35, 45, 55))
print(f"\n  weighted val S11 MAE: zero-shot {zs_weighted:.4f} -> "
      f"after 5 epochs {post_w:.4f} dB")
print("  PASS if it moved downward. No movement in 5 epochs is a red flag.")

with open(f'{DATA_ROOT}/artifacts/smoke_test_chunk12.json', 'w') as f:
    json.dump({'zero_shot_per_grid': zs,
               'zero_shot_weighted': float(zs_weighted),
               'true_s11_min_median': float(np.median(tmin)),
               'label_agreement': float(agree),
               'first_step_mse': float(first_loss),
               'grad_norm_median': float(np.median(norms)),
               'grad_norm_p90': float(np.percentile(norms, 90)),
               'frac_steps_clipped': float((norms > 1.0).mean()),
               'val_mae_after_5ep': float(post_w)}, f, indent=2)
print(f"\nSaved -> {DATA_ROOT}/artifacts/smoke_test_chunk12.json")

del model
torch.cuda.empty_cache()

CHECK 4 — loss scale and gradient clipping headroom


epoch 1/5:   0%|          | 0/63 [00:00<?, ?it/s]

  epoch 1: train MSE mean=0.5621 first=0.4776 last=0.3836 | grad-norm median=2.945 clipped=100%


epoch 2/5:   0%|          | 0/63 [00:00<?, ?it/s]

  epoch 2: train MSE mean=0.5165 first=0.8545 last=0.3103 | grad-norm median=2.227 clipped=100%


...

  epoch 4: train MSE mean=0.4980 first=0.4801 last=0.4546 | grad-norm median=2.027 clipped=100%


epoch 5/5:   0%|          | 0/63 [00:00<?, ?it/s]

  epoch 5: train MSE mean=0.4948 first=0.4074 last=0.2946 | grad-norm median=2.088 clipped=98%

  first-ever step MSE : 0.4776   (PASS if order ~1, not hundreds)
  grad-norm median    : 2.236   (PASS if <= ~1.0)
  grad-norm p90 / max : 4.177 / 12.658
  steps hitting clip  : 99.7%   (PASS if well under ~50%; near 100% => throttled, as in v1.0)


eval:   0%|          | 0/24 [00:00<?, ?it/s]


  weighted val S11 MAE: zero-shot 0.6717 -> after 5 epochs 0.3580 dB
  PASS if it moved downward. No movement in 5 epochs is a red flag.

Saved -> /content/drive/MyDrive/antenna_gnn/artifacts/smoke_test_chunk12.json
